# Complete PDF-to-Database Pipeline with OCR Integration

This notebook runs the complete pipeline from PDF files to extracted data in the database:

## Streamline Pipeline Overview

### Features:
- **arXiv Integration**: Download papers directly from arXiv using paper IDs
- **OCR Integration**: Process PDF files directly with Azure Mistral OCR
- **Smart Chunking**: Splits large documents using intelligent tokenization
- **Multi-Provider Support**: DeepSeek, Gemini, OpenAI analyzers
- **Database Integration**: Complete storage and retrieval system
- **Progressive Analysis**: Context-aware chunking for better results

## Setup and Import Required Libraries

In [1]:
import os
import sys
import json
import time
import base64
import requests
from pathlib import Path
from typing import Optional, List

# Add the current directory to sys.path to fix import issues
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

# Import arXiv downloader
from arxiv_tools.arxiv_downloader import download_arxiv_paper



# Import the tokenizer (this is key!)
from tokenizer import get_tokenizer

# Import the chunked analyzer and base analyzers
from analyser.base_analyser import BaseAnalyser
from analyser.chunked_analyser import ChunkedAnalyser, create_chunked_analyser
from analyser.deepseek_analyser import DeepSeekAnalyser
from analyser.gemini_analyser import GeminiAnalyser

# Import prompt templates - can switch between standard and robotics
from analyser import prompt_templates
from analyser import prompt_templates_robotics

# Import the db_integration module

# Import database integration
from analyser.db_integration import (
    analyze_and_store, 
    store_paper_from_md_file, 
    get_or_create_model, 
    store_extraction_results
)

# Set up environment variables if not already set
from dotenv import load_dotenv
load_dotenv()

print(" Libraries imported successfully")

 Libraries imported successfully


/Users/tobig/code/MasterThesis/OCR/code/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 0: arXiv Paper Download (Optional)

Download papers directly from arXiv using their paper IDs. This step is optional - you can skip it if you already have PDF files.

In [2]:
# arXiv Download Configuration
ARXIV_CONFIG = {
    'download_from_arxiv': True,   # Set to True to download from arXiv, False to use existing PDF
    'arxiv_id': '2402.07827',       # arXiv paper ID (e.g., '2101.12345' or '2101.12345v1')
    'download_dir': 'downloaded_papers',    # Directory to save downloaded papers
    'existing_pdf': 'sample_data/samplePDF.pdf'      # Path to existing PDF file (used if download_from_arxiv is False)
}

print(f" arXiv Download Configuration:")
print(f"  - Download from arXiv: {ARXIV_CONFIG['download_from_arxiv']}")
print(f"  - arXiv ID: {ARXIV_CONFIG['arxiv_id']}")
print(f"  - Download directory: {ARXIV_CONFIG['download_dir']}")
print(f"  - Existing PDF: {ARXIV_CONFIG['existing_pdf']}")

 arXiv Download Configuration:
  - Download from arXiv: True
  - arXiv ID: 2402.07827
  - Download directory: downloaded_papers
  - Existing PDF: sample_data/samplePDF.pdf


In [3]:
def download_arxiv_paper_for_pipeline(config):
    """Download paper from arXiv or use existing PDF."""
    
    print(f"\n=== Step 0: Paper Acquisition ===")
    
    if config['download_from_arxiv']:
        print(f" Downloading paper from arXiv: {config['arxiv_id']}")
        
        # Create download directory
        os.makedirs(config['download_dir'], exist_ok=True)
        
        # Download the paper
        start_time = time.time()
        pdf_path, metadata = download_arxiv_paper(config['arxiv_id'], config['download_dir'])
        elapsed_time = time.time() - start_time
        
        if pdf_path:
            print(f" Paper downloaded successfully in {elapsed_time:.2f} seconds")
            print(f"  - PDF path: {pdf_path}")
            print(f"  - Title: {metadata['title'][:100]}...")
            print(f"  - Authors: {', '.join(metadata['authors'][:3])}{'...' if len(metadata['authors']) > 3 else ''}")
            print(f"  - Published: {metadata['published'].strftime('%Y-%m-%d')}")
            print(f"  - Categories: {', '.join(metadata['categories'])}")
            
            # Get file size
            file_size_mb = os.path.getsize(pdf_path) / (1024 * 1024)
            print(f"  - File size: {file_size_mb:.2f} MB")
            
            return pdf_path, metadata
        else:
            print(f" Failed to download paper from arXiv after {elapsed_time:.2f} seconds")
            print(f"  - Check if arXiv ID '{config['arxiv_id']}' is valid")
            print(f"  - Check internet connection")
            return None, None
    else:
        print(f" Using existing PDF file: {config['existing_pdf']}")
        
        # Check if existing PDF exists
        if os.path.exists(config['existing_pdf']):
            file_size_mb = os.path.getsize(config['existing_pdf']) / (1024 * 1024)
            print(f" PDF file found")
            print(f"  - Path: {config['existing_pdf']}")
            print(f"  - Size: {file_size_mb:.2f} MB")
            
            # Create minimal metadata for existing file
            metadata = {
                'title': f"Local PDF: {Path(config['existing_pdf']).stem}",
                'authors': ['Unknown'],
                'published': None,
                'categories': ['Unknown'],
                'summary': 'Local PDF file'
            }
            
            return config['existing_pdf'], metadata
        else:
            print(f" PDF file not found: {config['existing_pdf']}")
            print(f"\n Available sample files:")
            sample_dir = 'sample_data'
            if os.path.exists(sample_dir):
                for file in os.listdir(sample_dir):
                    if file.endswith('.pdf'):
                        full_path = os.path.join(sample_dir, file)
                        size_mb = os.path.getsize(full_path) / (1024 * 1024)
                        print(f"  - {file} ({size_mb:.2f} MB)")
            else:
                print(f"  - No sample_data directory found")
            return None, None

# Download or locate the paper
#pdf_file_path, paper_metadata = download_arxiv_paper_for_pipeline(ARXIV_CONFIG)

    if pdf_file_path:
        print(f"\n Paper ready for processing: {pdf_file_path}")
    else:
        print(f"\n️ No paper available - please check configuration")

## Enhanced OCR Configuration and Functions

Configure Azure OCR API for PDF processing with detailed image annotations.


In [4]:
# Import enhanced OCR modules
from ocr.ocr_models import ImageAnnotation, ImageType, DataVisualizationType
from ocr.ocr_analysis import EnhancedOCRProcessor, process_pdf_with_enhanced_ocr

# Add standard OCR processing function for comparison
def process_pdf_with_ocr(pdf_path, output_dir, api_key=None, api_endpoint=None, timeout=120, verbose=True):
    """Standard OCR processing without image annotations (for comparison)."""
    
    if not api_key:
        api_key = os.getenv('AZURE_OCR_API_KEY', 'your_azure_ocr_api_key_here')
    if not api_endpoint:
        api_endpoint = os.getenv('AZURE_OCR_ENDPOINT', 'https://mistral-ocr-2503-zxbfx.eastus.models.ai.azure.com/v1/ocr')
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    pdf_name = Path(pdf_path).stem


    
    # Encode PDF
    try:
        with open(pdf_path, "rb") as pdf_file:
            file_content = pdf_file.read()
            base64_pdf = base64.b64encode(file_content).decode('utf-8')
    except Exception as e:
        if verbose:
            print(f" Error encoding PDF: {e}")
        return None, None
    
    # Standard OCR payload (no annotations)
    payload = {
        "model": "mistral-ocr-latest",
        "document": {
            "document_url": f"data:application/pdf;base64,{base64_pdf}",
            "type": "document_url"
        }
    }
    
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {api_key}"
    }
    
    try:
        if verbose:
            print("  - Sending standard OCR request to Azure API...")
        
        start_time = time.time()
        response = requests.post(
            api_endpoint,
            headers=headers,
            json=payload,
            timeout=timeout
        )
        elapsed_time = time.time() - start_time
        
        if verbose:
            print(f"  - API response received in {elapsed_time:.2f}s (Status: {response.status_code})")
        
        if response.status_code == 200:
            result = response.json()

            print(f"Response status: {response.status_code}")
            print(f"Response headers: {dict(response.headers)}")
            print(f"Response text length: {len(response.text)}")
            print(f"Response text preview: {response.text[:500]}")
            
            if result is None:
                if verbose:
                    print(" OCR API returned None response")
                return None, None
            
            # Extract text content
            extracted_text = ""
            if 'pages' in result and result['pages']:
                for page_idx, page in enumerate(result['pages']):
                    if page and 'markdown' in page:
                        extracted_text += f"\n\n--- Page {page_idx + 1} ---\n\n"
                        extracted_text += page['markdown'] + "\n\n"
            
            # Save files
            md_path = os.path.join(output_dir, f"{pdf_name}.md")
            json_path = os.path.join(output_dir, f"{pdf_name}_ocr_response.json")
            
            with open(md_path, 'w', encoding='utf-8') as f:
                f.write(extracted_text)
            
            with open(json_path, 'w', encoding='utf-8') as f:
                json.dump(result, f, indent=2, ensure_ascii=False)
            
            if verbose:
                print(f" Standard OCR processing complete:")
                print(f"  - Markdown: {md_path}")
                print(f"  - JSON: {json_path}")
                print(f"  - Text length: {len(extracted_text):,} characters")
            
            return md_path, json_path
        else:
            if verbose:
                print(f" OCR API error: {response.status_code}")
                print(f"  Response: {response.text[:200]}...")
            return None, None
            
    except Exception as e:
        if verbose:
            print(f" OCR processing error: {e}")
        return None, None

# OCR Configuration - Enhanced for Images Only
ENHANCED_OCR_CONFIG = {
    'api_key': os.getenv('AZURE_OCR_API_KEY', 'your_azure_ocr_api_key_here'),
    'api_endpoint': os.getenv('AZURE_OCR_ENDPOINT', 'https://mistral-ocr-2503-zxbfx.eastus.models.ai.azure.com/v1/ocr'),
    'timeout': 180,  # Increased timeout for annotation processing
    'max_annotation_pages': 8,  # API limit for detailed annotations
    'include_images': False,  # Don't include base64 images when using detailed annotations
    'use_annotations': False,  # SET TO FALSE IF ENDPOINT DOESN'T SUPPORT ANNOTATIONS
    'verbose': True
}

def run_enhanced_ocr_processing(config):
    """Run enhanced OCR processing with optional detailed image annotations."""
    
    print(f"\n=== Step 1: Enhanced OCR Processing ===")
    
    if not config['process_pdf']:
        print(" OCR processing disabled in config")
        return None, None, None
    
    pdf_path = config['pdf_path']
    
    # Check if PDF exists
    if not os.path.exists(pdf_path):
        print(f" PDF file not found: {pdf_path}")
        return None, None, None
    
    # Get file info
    file_size_mb = os.path.getsize(pdf_path) / (1024 * 1024)
    print(f" Input PDF: {pdf_path} ({file_size_mb:.2f} MB)")
    
    ocr_mode = "Enhanced OCR with bbox annotations" if ENHANCED_OCR_CONFIG['use_annotations'] else "Standard OCR"
    print(f" Processing mode: {ocr_mode}")
    if ENHANCED_OCR_CONFIG['use_annotations']:
        print(f"  - Primary: Enhanced OCR with detailed image annotations")
        print(f"  - Fallback: Standard OCR if enhanced fails")
    else:
        print(f"  - Using: Standard OCR (annotations disabled in config)")
    
    # Run OCR processing
    start_time = time.time()
    md_path, json_path, annotations = process_pdf_with_enhanced_ocr(
        pdf_path=pdf_path,
        output_dir=config['output_dir'],
        api_key=ENHANCED_OCR_CONFIG['api_key'],
        api_endpoint=ENHANCED_OCR_CONFIG['api_endpoint'],
        max_pages=ENHANCED_OCR_CONFIG['max_annotation_pages'],
        include_images=ENHANCED_OCR_CONFIG['include_images'],
        use_annotations=ENHANCED_OCR_CONFIG['use_annotations'],  # Pass the flag
        verbose=ENHANCED_OCR_CONFIG['verbose']
    )
    elapsed_time = time.time() - start_time
    
    if md_path and json_path:
        if annotations and ENHANCED_OCR_CONFIG['use_annotations']:
            print(f"\n Enhanced OCR with annotations completed in {elapsed_time:.2f} seconds")
            print(f"  - Markdown file: {md_path}")
            print(f"  - JSON response: {json_path}")
            print(f"  - Total image annotations: {annotations['total_annotations']}")
            
            # Display annotation summary
            if annotations['total_annotations'] > 0:
                print(f"\n Image Annotation Summary:")
                print(f"\n Annotation Types:")
                for img_type, count in annotations.get('annotation_types', {}).items():
                    print(f"  - {img_type.title()}: {count}")
                
                print(f"\n By Page:")
                for page_num, page_annotations in annotations['annotations_by_page'].items():
                    print(f"  Page {page_num}: {len(page_annotations)} annotations")
                    for i, annotation in enumerate(page_annotations[:2]):  # Show first 2 per page
                        img_type = annotation.get('image_type', 'Unknown')
                        title = annotation.get('title', 'Untitled')[:40]
                        print(f"    {i+1}. {img_type}: {title}...")
                    if len(page_annotations) > 2:
                        print(f"    ... and {len(page_annotations) - 2} more")
        else:
            ocr_type = "Standard OCR" if not ENHANCED_OCR_CONFIG['use_annotations'] else "Standard OCR fallback"
            print(f"\n {ocr_type} completed in {elapsed_time:.2f} seconds")
            print(f"  - Markdown file: {md_path}")
            print(f"  - JSON response: {json_path}")
            if not ENHANCED_OCR_CONFIG['use_annotations']:
                print(f"  - Note: Enhanced annotations disabled in config")
            else:
                print(f"  - Note: Enhanced annotations not available (used fallback)")
        
        # Check the generated markdown
        with open(md_path, 'r', encoding='utf-8') as f:
            content = f.read()
        print(f"  - Generated text: {len(content):,} characters")
        print(f"  - Preview: {content[:300]}...")
        
        return md_path, json_path, annotations
    else:
        print(f" OCR processing failed after {elapsed_time:.2f} seconds")
        return None, None, None


    """Run enhanced OCR processing with optional detailed image annotations."""
    
    print(f"\n=== Step 1: Enhanced OCR Processing ===")
    
    if not config['process_pdf']:
        print(" OCR processing disabled in config")
        return None, None, None
    
    pdf_path = config['pdf_path']
    
    # Check if PDF exists
    if not os.path.exists(pdf_path):
        print(f" PDF file not found: {pdf_path}")
        return None, None, None
    
    # Get file info
    file_size_mb = os.path.getsize(pdf_path) / (1024 * 1024)
    print(f" Input PDF: {pdf_path} ({file_size_mb:.2f} MB)")
    
    ocr_mode = "Enhanced OCR with bbox annotations" if ENHANCED_OCR_CONFIG['use_annotations'] else "Standard OCR"
    print(f" Processing mode: {ocr_mode}")
    if ENHANCED_OCR_CONFIG['use_annotations']:
        print(f"  - Primary: Enhanced OCR with detailed image annotations")
        print(f"  - Fallback: Standard OCR if enhanced fails")
    else:
        print(f"  - Using: Standard OCR (annotations disabled in config)")
    
    # Run OCR processing
    start_time = time.time()
    md_path, json_path, annotations = process_pdf_with_enhanced_ocr(
        pdf_path=pdf_path,
        output_dir=config['output_dir'],
        api_key=ENHANCED_OCR_CONFIG['api_key'],
        api_endpoint=ENHANCED_OCR_CONFIG['api_endpoint'],
        max_pages=ENHANCED_OCR_CONFIG['max_annotation_pages'],
        include_images=ENHANCED_OCR_CONFIG['include_images'],
        use_annotations=ENHANCED_OCR_CONFIG['use_annotations'],  # Pass the flag
        verbose=ENHANCED_OCR_CONFIG['verbose']
    )
    elapsed_time = time.time() - start_time
    
    if md_path and json_path:
        if annotations and ENHANCED_OCR_CONFIG['use_annotations']:
            print(f"\n Enhanced OCR with annotations completed in {elapsed_time:.2f} seconds")
            print(f"  - Markdown file: {md_path}")
            print(f"  - JSON response: {json_path}")
            print(f"  - Total image annotations: {annotations['total_annotations']}")
            
            # Display annotation summary
            if annotations['total_annotations'] > 0:
                print(f"\n Image Annotation Summary:")
                print(f"\n Annotation Types:")
                for img_type, count in annotations.get('annotation_types', {}).items():
                    print(f"  - {img_type.title()}: {count}")
                
                print(f"\n By Page:")
                for page_num, page_annotations in annotations['annotations_by_page'].items():
                    print(f"  Page {page_num}: {len(page_annotations)} annotations")
                    for i, annotation in enumerate(page_annotations[:2]):  # Show first 2 per page
                        img_type = annotation.get('image_type', 'Unknown')
                        title = annotation.get('title', 'Untitled')[:40]
                        print(f"    {i+1}. {img_type}: {title}...")
                    if len(page_annotations) > 2:
                        print(f"    ... and {len(page_annotations) - 2} more")
        else:
            ocr_type = "Standard OCR" if not ENHANCED_OCR_CONFIG['use_annotations'] else "Standard OCR fallback"
            print(f"\n {ocr_type} completed in {elapsed_time:.2f} seconds")
            print(f"  - Markdown file: {md_path}")
            print(f"  - JSON response: {json_path}")
            if not ENHANCED_OCR_CONFIG['use_annotations']:
                print(f"  - Note: Enhanced annotations disabled in config")
            else:
                print(f"  - Note: Enhanced annotations not available (used fallback)")
        
        # Check the generated markdown
        with open(md_path, 'r', encoding='utf-8') as f:
            content = f.read()
        print(f"  - Generated text: {len(content):,} characters")
        print(f"  - Preview: {content[:300]}...")
        
        return md_path, json_path, annotations
    else:
        print(f" OCR processing failed after {elapsed_time:.2f} seconds")
        return None, None, None

# Configuration for complete pipeline
pdf_file_path, paper_metadata = download_arxiv_paper_for_pipeline(ARXIV_CONFIG)

# PROMPT TEMPLATE CONFIGURATION
# Set to True to use robotics-specific prompts, False for standard prompts
USE_ROBOTICS_PROMPTS = False  # <<<--- CHANGE THIS TO SWITCH PROMPT TEMPLATES

CONFIG = {
    # Input files - now uses the downloaded/selected PDF
    'pdf_path': pdf_file_path if pdf_file_path else 'sample_data/samplePDF.pdf',
    'paper_path': None,  # Will be set after OCR processing
    'output_dir': f'results/{ARXIV_CONFIG["arxiv_id"].replace("/", "_")}' if ARXIV_CONFIG['download_from_arxiv'] else 'results/complete_pipeline',
    
    # Paper metadata from arXiv or local file
    'paper_metadata': paper_metadata,
    
    # Analysis settings
    'provider': 'deepseek',  # 'deepseek', 'gemini', or 'openai'
    'max_tokens': 6000,  # Maximum tokens per chunk
    'overlap_tokens': 500,  # Overlap between chunks
    'progressive': True,  # Use progressive chunking
    'exclude_sections': ['References'],  # Sections to exclude
    
    # Prompt template type
    'use_robotics_prompts': USE_ROBOTICS_PROMPTS,
    
    # API tokens
    'deepseek_token': os.getenv('DEEPSEEK_API_TOKEN'),
    'gemini_token': os.getenv('GEMINI_API_KEY'),
    
    # Database options
    'save_to_db': True,  # Whether to save results to database
    'arxiv_id': ARXIV_CONFIG['arxiv_id'] if ARXIV_CONFIG['download_from_arxiv'] else 'pipeline-test-001',
    'title': paper_metadata['title'] if paper_metadata else 'Complete Pipeline Test Paper',
    
    # OCR options
    'process_pdf': True,  # Whether to run OCR on PDF first
    'save_images': True   # Whether to save extracted images
}

# Create output directory
os.makedirs(CONFIG['output_dir'], exist_ok=True)

print(f" Complete Pipeline Configuration:")
for key, value in CONFIG.items():
    if 'token' in key.lower():
        print(f"  {key}: {' Set' if value else ' Not set'}")
    elif key == 'paper_metadata':
        print(f"  {key}: {' Available' if value else ' Not available'}")
    else:
        print(f"  {key}: {value}")

2025-10-04 17:36:47,092 - arxiv - INFO - Requesting page (first: True, try: 0): https://export.arxiv.org/api/query?search_query=&id_list=2402.07827&sortBy=relevance&sortOrder=descending&start=0&max_results=100



=== Step 0: Paper Acquisition ===


2025-10-04 17:36:47,414 - arxiv - INFO - Got first page: 1 of 1 total results


Downloaded paper: Aya Model: An Instruction Finetuned Open-Access Multilingual Language Model
Saved to: downloaded_papers/2402.07827.pdf
 Paper downloaded successfully in 0.69 seconds
  - PDF path: downloaded_papers/2402.07827.pdf
  - Title: Aya Model: An Instruction Finetuned Open-Access Multilingual Language Model...
  - Authors: Ahmet Üstün, Viraat Aryabumi, Zheng-Xin Yong...
  - Published: 2024-02-12
  - Categories: cs.CL
  - File size: 3.65 MB
 Complete Pipeline Configuration:
  pdf_path: downloaded_papers/2402.07827.pdf
  paper_path: None
  output_dir: results/2402.07827
  paper_metadata:  Available
  provider: deepseek
  max_tokens:  Set
  overlap_tokens:  Set
  progressive: True
  exclude_sections: ['References']
  use_robotics_prompts: False
  deepseek_token:  Set
  gemini_token:  Set
  save_to_db: True
  arxiv_id: 2402.07827
  title: Aya Model: An Instruction Finetuned Open-Access Multilingual Language Model
  process_pdf: True
  save_images: True


## Step 1: OCR Processing (PDF → Markdown)

Convert PDF file to markdown text using Azure OCR API.

In [5]:
def run_ocr_processing(config):
    """Run OCR processing on the input PDF file."""
    
    print(f"\n=== Step 1: OCR Processing ===")
    
    if not config['process_pdf']:
        print(" OCR processing disabled in config")
        return None, None
    
    pdf_path = config['pdf_path']
    
    # Check if PDF exists
    if not os.path.exists(pdf_path):
        print(f" PDF file not found: {pdf_path}")
        print("\n Available sample files:")
        sample_dir = 'sample_data'
        if os.path.exists(sample_dir):
            for file in os.listdir(sample_dir):
                if file.endswith('.pdf'):
                    full_path = os.path.join(sample_dir, file)
                    size_kb = os.path.getsize(full_path) / 1024
                    print(f"  - {file} ({size_kb:.1f} KB)")
        return None, None
    
    # Get file info
    file_size_kb = os.path.getsize(pdf_path) / 1024
    print(f" Input PDF: {pdf_path} ({file_size_kb:.2f} KB)")
    
    # Process with OCR
    start_time = time.time()
    md_path, json_path = process_pdf_with_ocr(pdf_path, config['output_dir'])
    elapsed_time = time.time() - start_time
    
    if md_path and json_path:
        print(f"\n OCR processing completed in {elapsed_time:.2f} seconds")
        print(f"  - Markdown file: {md_path}")
        print(f"  - JSON response: {json_path}")
        
        # Check the generated markdown
        with open(md_path, 'r', encoding='utf-8') as f:
            content = f.read()
        print(f"  - Generated text: {len(content):,} characters")
        print(f"  - Preview: {content[:200]}...")
        
        return md_path, json_path
    else:
        print(f" OCR processing failed after {elapsed_time:.2f} seconds")
        return None, None

# Run OCR processing
#md_file, json_file = run_ocr_processing(CONFIG)
print("Skipping OCR processing for now...")

Skipping OCR processing for now...


## Step 2: Token Analysis

Analyze the markdown file to determine if chunking is needed.

In [6]:
def analyze_document_tokens(paper_path, provider='deepseek', max_tokens=8192):
    """Analyze document tokens to determine chunking needs."""
    
    print(f"\n=== Step 2: Token Analysis ===")
    
    if not paper_path or not os.path.exists(paper_path):
        print(f" Paper not found: {paper_path}")
        return None, None
    
    # Get file info
    file_size_kb = os.path.getsize(paper_path) / 1024
    print(f" Document: {paper_path} ({file_size_kb:.2f} KB)")
    
    # Count tokens using the tokenizer
    tokenizer = get_tokenizer(provider, max_tokens=max_tokens)
    if not tokenizer:
        print(f" Could not create tokenizer for provider: {provider}")
        return None, None
    
    try:
        with open(paper_path, 'r', encoding='utf-8') as f:
            content = f.read()
        
        token_count = tokenizer.count_tokens(content)
        print(f" Token Analysis:")
        print(f"  - Content length: {len(content):,} characters")
        print(f"  - Token count: {token_count:,} tokens")
        print(f"  - Tokenizer max: {tokenizer.max_tokens:,} tokens")
        
        # Calculate effective limits
        prompt_buffer = 1000  # Tokens for system prompt
        response_buffer = 1000  # Tokens for response
        effective_limit = tokenizer.max_tokens - prompt_buffer - response_buffer
        
        needs_chunking = token_count > effective_limit
        
        print(f"  - Effective limit: {effective_limit:,} tokens")
        print(f"  - Chunking needed: {'Yes' if needs_chunking else 'No'}")
        
        if needs_chunking:
            estimated_chunks = (token_count // CONFIG['max_tokens']) + 1
            print(f"  - Estimated chunks: {estimated_chunks}")
        
        return token_count, needs_chunking
        
    except Exception as e:
        print(f" Error analyzing tokens: {e}")
        return None, None

# Analyze document tokens
if CONFIG['paper_path']:
    token_count, needs_chunking = analyze_document_tokens(CONFIG['paper_path'], CONFIG['provider'])
else:
    print("️ No paper path available for token analysis")
    token_count, needs_chunking = None, None

️ No paper path available for token analysis


## PREPARE END-2-END PIPELINE FUNCTION

This function combines all pipeline steps into a single, comprehensive workflow that:
1. Downloads papers from arXiv (or uses existing PDFs)
2. Processes PDFs with OCR to extract text and images
3. Analyzes token count and applies chunking if needed
4. Runs LLM analysis with multiple providers and runs
5. Stores all results in the database
6. Provides detailed progress reporting and error handling

In [7]:
def run_complete_pipeline(
    # Paper source configuration
    arxiv_id: Optional[str] = None,
    pdf_path: Optional[str] = None,
    download_dir: str = "downloaded_papers",
    
    # Enhanced OCR configuration
    run_ocr: bool = True,
    use_enhanced_ocr: bool = True,  # Enhanced image annotations
    use_annotations: bool = False,  # NEW: Control annotation usage
    ocr_output_dir: Optional[str] = None,
    max_annotation_pages: int = 8,
    force_reprocess: bool = False,  # NEW: Force redownload/reprocess even if MD exists
    
    # Analysis configuration
    run_analysis: bool = True,
    deepseek_runs: int = 1,
    gemini_runs: int = 0,
    openai_runs: int = 0,
    
    # Prompt template configuration
    use_robotics_prompts: bool = False,  # NEW: Switch to robotics-specific prompts
    
    # Advanced settings
    temperature: float = 0.3,
    max_chunk_size: int = 15000,
    chunk_overlap: int = 200,
    
    # Progress reporting
    verbose: bool = True
) -> dict:
    """
    Run the complete PDF-to-Database pipeline with enhanced image annotations.
    
    Enhanced OCR focuses specifically on visual elements:
    - Detailed analysis of images, graphs, charts, tables
    - Data extraction from visualizations
    - Technical details and scientific information
    - Context relevance for each visual element
    
    Args:
        use_enhanced_ocr: Whether to use enhanced OCR with detailed image annotations
        max_annotation_pages: Maximum pages to process for detailed annotations (API limit: 8)
        force_reprocess: If False, check for existing MD files before downloading/OCR processing
    """
    
    pipeline_results = {
        'success': False,
        'paper_id': None,
        'pdf_path': None,
        'md_path': None,
        'ocr_results': None,
        'image_annotations': None,  # Focused on images only
        'analysis_results': {
            'deepseek': [],
            'gemini': [],
            'openai': []
        },
        'images_stored': 0,
        'image_annotations_count': 0,  # Count of detailed image annotations
        'total_extractions': 0,
        'processing_time': 0,
        'errors': [],
        'skipped_download': False,  # NEW: Track if download was skipped
        'skipped_ocr': False,       # NEW: Track if OCR was skipped
        'existing_md_used': None   # NEW: Path to existing MD file if used
    }
    
    import time
    import os
    from pathlib import Path
    
    start_time = time.time()

    config = {
        'exclude_sections': ['References'],  # Sections to exclude
    }
    
    # Select prompt templates based on configuration
    if use_robotics_prompts:
        if verbose:
            print(f" Using robotics-specific prompt templates")
        template_module = prompt_templates_robotics
    else:
        if verbose:
            print(f" Using standard prompt templates")
        template_module = prompt_templates
    
    try:
        # Validate input parameters
        if not arxiv_id and not pdf_path:
            raise ValueError("Either arxiv_id or pdf_path must be provided")
        
        if arxiv_id and pdf_path:
            if verbose:
                print("️  Both arxiv_id and pdf_path provided, using arxiv_id")
        
        # === STEP 0: Check for existing markdown files ===
        existing_md_path = None
        if not force_reprocess:
            # Determine expected markdown file path
            paper_identifier = arxiv_id or Path(pdf_path).stem
            if not ocr_output_dir:
                ocr_output_dir = os.path.join("results", paper_identifier.replace('/', '_'))
            
            # Check for existing markdown file
            expected_md_name = f"{paper_identifier.replace('/', '_')}.md"
            expected_md_path = os.path.join(ocr_output_dir, expected_md_name)
            
            if os.path.exists(expected_md_path):
                # Check file size to ensure it's not empty
                file_size = os.path.getsize(expected_md_path)
                if file_size > 1000:  # At least 1KB
                    existing_md_path = expected_md_path
                    if verbose:
                        print(f"\n Found existing markdown file: {expected_md_path}")
                        print(f"   Size: {file_size / 1024:.1f} KB")
                        print(f"   Skipping download and OCR processing...")
                    
                    pipeline_results['skipped_download'] = True
                    pipeline_results['skipped_ocr'] = True
                    pipeline_results['existing_md_used'] = existing_md_path
                    pipeline_results['md_path'] = existing_md_path
                elif verbose:
                    print(f"\n️ Found existing markdown file but it's too small ({file_size} bytes), will reprocess")
            elif verbose:
                print(f"\n No existing markdown file found at: {expected_md_path}")
                print(f"   Will proceed with download and OCR processing...")
        elif verbose:
            print(f"\n Force reprocess enabled - will download and process regardless of existing files")
        
        # === STEP 1: Paper Acquisition ===
        if verbose:
            print("\n" + "="*60)
            print(" STARTING COMPLETE PIPELINE")
            print("="*60)
        
        if existing_md_path:
            # Skip download, use existing markdown
            if verbose:
                print(f"\n Step 1: Using existing markdown file")
            
            # Create minimal metadata for existing file
            paper_title = Path(existing_md_path).stem
            metadata = {
                'title': paper_title,
                'authors': ['Unknown'],
                'published': None,
                'categories': [],
                'summary': ''
            }
            # Try to find corresponding PDF if it exists
            if arxiv_id:
                potential_pdf = os.path.join(download_dir, f"{arxiv_id}.pdf")
                if os.path.exists(potential_pdf):
                    pipeline_results['pdf_path'] = potential_pdf
            elif pdf_path and os.path.exists(pdf_path):
                pipeline_results['pdf_path'] = pdf_path
        elif arxiv_id:
            temp_config = ARXIV_CONFIG.copy()
            temp_config['arxiv_id'] = arxiv_id
            temp_config['download_dir'] = download_dir
            temp_config['existing_pdf'] = None
            
            downloaded_path, metadata = download_arxiv_paper_for_pipeline(temp_config)
            
            if not downloaded_path:
                raise Exception(f"Failed to download paper from arXiv: {arxiv_id}")
            
            pipeline_results['pdf_path'] = downloaded_path
            paper_title = metadata.get('title', f'arXiv-{arxiv_id}')
            
            if verbose:
                print(f" Downloaded: {paper_title[:80]}...")
                print(f"   Size: {os.path.getsize(downloaded_path) / (1024*1024):.1f} MB")
        else:
            if verbose:
                print(f"\n Step 1: Using existing PDF: {pdf_path}")
            
            if not os.path.exists(pdf_path):
                raise FileNotFoundError(f"PDF file not found: {pdf_path}")
            
            pipeline_results['pdf_path'] = pdf_path
            paper_title = Path(pdf_path).stem
            
            # Create minimal metadata
            metadata = {
                'title': paper_title,
                'authors': ['Unknown'],
                'published': None,
                'categories': [],
                'summary': ''
            }
        
        # === STEP 2: Enhanced OCR Processing ===
        if run_ocr and not existing_md_path:
            if verbose:
                ocr_type = "Enhanced Image Annotations" if use_enhanced_ocr else "Standard OCR"
                print(f"\n Step 2: {ocr_type}")

            
            
            # Set up configuration for OCR
            paper_identifier = arxiv_id or Path(pdf_path).stem
            if not ocr_output_dir:
                ocr_output_dir = os.path.join("results", paper_identifier.replace('/', '_'))
            
            if use_enhanced_ocr:
                # Use enhanced OCR with optional detailed image annotations
                md_path, json_path, annotations = process_pdf_with_enhanced_ocr(
                    pdf_path=pipeline_results['pdf_path'],
                    output_dir=ocr_output_dir,
                    max_pages=max_annotation_pages,
                    include_images=False,  # Use annotations instead of images
                    use_annotations=use_annotations,  # Pass the flag
                    verbose=verbose
                    
                )
                
                if annotations and use_annotations:
                    pipeline_results['image_annotations'] = annotations
                    pipeline_results['image_annotations_count'] = annotations.get('total_annotations', 0)
                    
                    if verbose:
                        print(f"  - Image annotations: {pipeline_results['image_annotations_count']}")
                        # Show annotation breakdown by type
                        annotation_types = annotations.get('annotation_types', {})
                        for img_type, count in sorted(annotation_types.items()):
                            print(f"    - {img_type}: {count}")
                elif verbose and use_annotations:
                    print(f"  - Image annotations: 0 (fallback to standard OCR used)")
                elif verbose:
                    print(f"  - Image annotations: disabled in config")
            else:
                # Use standard OCR processing
                temp_config = {
                    'pdf_path': pipeline_results['pdf_path'],
                    'output_dir': ocr_output_dir,
                    'process_pdf': True,
                    'save_images': True
                }
                md_path, json_path = run_ocr_processing(temp_config)
            
            if not md_path:
                raise Exception("OCR processing failed")
            
            pipeline_results['md_path'] = md_path
            pipeline_results['ocr_results'] = json_path
            
            if verbose:
                print(f" OCR completed")
                print(f"   Markdown saved: {md_path}")
                if use_enhanced_ocr and use_annotations:
                    print(f"   Image annotations: {pipeline_results['image_annotations_count']}")
                elif use_enhanced_ocr:
                    print(f"   Image annotations: disabled")
        elif existing_md_path:
            if verbose:
                print(f"\n Step 2: Skipping OCR (using existing markdown)")
        elif not run_ocr:
            if verbose:
                print(f"\n Step 2: OCR processing disabled")
        
        # === STEP 3: Database Storage (Paper) ===
        if verbose:
            print(f"\n Step 3: Storing paper in database")
        
        paper_id = store_paper_from_md_file(
            pipeline_results['md_path'],
            arxiv_id=arxiv_id or Path(pdf_path).stem,
            title=metadata['title']
        )
        
        if not paper_id:
            raise Exception("Failed to store paper in database")
        
        pipeline_results['paper_id'] = paper_id
        
        if verbose:
            print(f" Paper stored with ID: {paper_id}")
        
        # === STEP 4: LLM Analysis ===
        if run_analysis and pipeline_results['md_path']:
            if verbose:
                print(f"\n Step 4: LLM Analysis")
            
            total_runs = deepseek_runs + gemini_runs + openai_runs
            if verbose:
                print(f"   Planning {total_runs} total analysis runs")
                print(f"   DeepSeek: {deepseek_runs}, Gemini: {gemini_runs}, OpenAI: {openai_runs}")
            
            # Use the existing token analysis function
            token_count, needs_chunking = analyze_document_tokens(
                pipeline_results['md_path'], 
                provider='deepseek',  # Use DeepSeek for token analysis <<<--- change this if needed
                max_tokens=max_chunk_size
            )
            
            if verbose:
                print(f"   Document tokens: {token_count:,}")
                print(f"   Needs chunking: {'Yes' if needs_chunking else 'No'}")
            
            # DeepSeek Analysis
            if deepseek_runs > 0:
                if verbose:
                    print(f"\n    Running {deepseek_runs} DeepSeek analysis(es)")
                
                for run_idx in range(deepseek_runs):
                    try:
                        run_temp = temperature + (run_idx * 0.05)  # Slight variation
                        
                        # Use the existing analyze_and_store function directly
                        if needs_chunking:
                            chunked_analyser = create_chunked_analyser(
                                base_analyser=DeepSeekAnalyser(prompt_template=template_module.PUBLICATION_ANALYSIS_TEMPLATE),
                                provider='deepseek',
                                max_tokens=max_chunk_size,
                                progressive=True,
                                template_module=template_module
                            )
                            if verbose:
                                print(f"      Using chunked analyzer for DeepSeek (Run {run_idx + 1})")

                                        # Set excluded sections
                            if hasattr(chunked_analyser, 'exclude_sections') and config['exclude_sections']:
                                chunked_analyser.exclude_sections = config['exclude_sections']
                                print(f"  - Excluding sections: {', '.join(config['exclude_sections'])}")

                            analyzer = chunked_analyser
                        else:
                            analyzer = DeepSeekAnalyser(prompt_template=template_module.PUBLICATION_ANALYSIS_TEMPLATE)
                        
                        if verbose:
                            print(f"      Running DeepSeek analysis (Run {run_idx + 1}) with temperature {run_temp:.2f}")

                        paper_id_result, model_id, run_ids = analyze_and_store(
                            analyzer,
                            pipeline_results['md_path'],
                            "deepseek-chat",
                            "DeepSeek",
                            arxiv_id=arxiv_id,
                            temperature=run_temp
                        )
                        
                        if run_ids:
                            pipeline_results['analysis_results']['deepseek'].extend(run_ids)
                            pipeline_results['total_extractions'] += len(run_ids)
                            
                            if verbose:
                                print(f"      Run {run_idx + 1}: {len(run_ids)} extractions")
                        else:
                            if verbose:
                                print(f"      Run {run_idx + 1}: Failed")
                            pipeline_results['errors'].append(f"DeepSeek run {run_idx + 1} failed")
                    
                    except Exception as e:
                        error_msg = f"DeepSeek run {run_idx + 1} error: {str(e)}"
                        pipeline_results['errors'].append(error_msg)
                        if verbose:
                            print(f"      Run {run_idx + 1}: {str(e)}")
            
            # Gemini Analysis
            if gemini_runs > 0:
                if verbose:
                    print(f"\n    Running {gemini_runs} Gemini analysis(es)")
                
                for run_idx in range(gemini_runs):
                    try:
                        
                        run_temp = temperature + (run_idx * 0.05)  # Slight variation
                        
                        # Use the existing analyze_and_store function directly
                        if needs_chunking:
                            chunked_analyser = create_chunked_analyser(
                                base_analyser=GeminiAnalyser(prompt_template=template_module.PUBLICATION_ANALYSIS_TEMPLATE),
                                provider='gemini',
                                max_tokens=max_chunk_size,
                                progressive=True,
                                template_module=template_module
                            )
                            if verbose:
                                print(f"      Using chunked analyzer for Gemini (Run {run_idx + 1})")

                                        # Set excluded sections
                            if hasattr(chunked_analyser, 'exclude_sections') and config['exclude_sections']:
                                chunked_analyser.exclude_sections = config['exclude_sections']
                                print(f"  - Excluding sections: {', '.join(config['exclude_sections'])}")

                            analyzer = chunked_analyser
                        else:
                            analyzer = GeminiAnalyser(prompt_template=template_module.PUBLICATION_ANALYSIS_TEMPLATE)

                        if verbose:
                            print(f"      Running Gemini analysis (Run {run_idx + 1}) with temperature {run_temp:.2f}")

                        paper_id_result, model_id, run_ids = analyze_and_store(
                            analyzer,
                            pipeline_results['md_path'],
                            "gemini-pro",
                            "Google",
                            arxiv_id=arxiv_id,
                            temperature=run_temp
                        )
                        
                        
                        if run_ids:
                            pipeline_results['analysis_results']['gemini'].extend(run_ids)
                            pipeline_results['total_extractions'] += len(run_ids)
                            
                            if verbose:
                                print(f"      Run {run_idx + 1}: {len(run_ids)} extractions")
                        else:
                            if verbose:
                                print(f"      Run {run_idx + 1}: Failed")
                            pipeline_results['errors'].append(f"Gemini run {run_idx + 1} failed")
                    
                    except Exception as e:
                        error_msg = f"Gemini run {run_idx + 1} error: {str(e)}"
                        pipeline_results['errors'].append(error_msg)
                        if verbose:
                            print(f"      Run {run_idx + 1}: {str(e)}")
        
        # === PIPELINE COMPLETION ===
        pipeline_results['success'] = True
        pipeline_results['processing_time'] = time.time() - start_time
        
        if verbose:
            print(f"\n" + "="*60)
            print(f" ENHANCED PIPELINE COMPLETED SUCCESSFULLY")
            print(f"="*60)
            print(f" Summary:")
            print(f"   Paper ID: {pipeline_results['paper_id']}")
            print(f"   Processing time: {pipeline_results['processing_time']:.1f}s")
            if pipeline_results['skipped_download']:
                print(f"   ⏭️  Skipped download (existing file used)")
            if pipeline_results['skipped_ocr']:
                print(f"   ⏭️  Skipped OCR (existing markdown used)")
            if use_enhanced_ocr and use_annotations:
                print(f"   Image annotations: {pipeline_results['image_annotations_count']}")
            print(f"   Total extractions: {pipeline_results['total_extractions']}")
        
        return pipeline_results
        
    except Exception as e:
        pipeline_results['success'] = False
        pipeline_results['processing_time'] = time.time() - start_time
        error_msg = f"Pipeline failed: {str(e)}"
        pipeline_results['errors'].append(error_msg)
        
        if verbose:
            print(f"\n PIPELINE FAILED after {pipeline_results['processing_time']:.1f}s")
            print(f"Error: {str(e)}")
        
        return pipeline_results

print(" Enhanced integrated pipeline function updated with existing file check!")

 Enhanced integrated pipeline function updated with existing file check!


## SETUP THE PIPELINE

In [8]:
# This code imports arxiv ids from the data/arxiv_ids/arxivIDs.txt file and and saves them in a list.
def import_arxiv_ids(file_path):
    """Import arXiv IDs from a text file."""
    arxiv_ids = []
    try:
        with open(file_path, 'r') as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith('#'):  # Ignore empty lines and comments
                    arxiv_ids.append(line)
        print(f" Imported {len(arxiv_ids)} arXiv IDs from {file_path}")

        #remove duplicates
        arxiv_ids = list(set(arxiv_ids))
        print(f" Removed duplicates, {len(arxiv_ids)} unique arXiv IDs available.")
    except Exception as e:
        print(f" Error importing arXiv IDs: {e}")
    return arxiv_ids

In [9]:
# Example usage of the complete pipeline function with improved parallel processing
print("\nSETTING UP COMPLETE PIPELINE WITH PARALLEL PROCESSING...")
print("="*80)  
from concurrent.futures import ThreadPoolExecutor, as_completed
from IPython.display import clear_output
import time

def download_papers_sequentially(arxiv_ids, download_dir="downloaded_papers"):
    """Download all papers sequentially to avoid arXiv rate limiting."""
    print(f"\n Phase 1: Sequential Download of {len(arxiv_ids)} papers")
    print("-" * 60)
    
    download_results = {}
    successful_downloads = 0
    
    for i, arxiv_id in enumerate(arxiv_ids, 1):
        print(f"\n Downloading {i}/{len(arxiv_ids)}: {arxiv_id}", end=" ... ")
        
        try:
            # Check if already downloaded
            expected_pdf = os.path.join(download_dir, f"{arxiv_id}.pdf")
            if os.path.exists(expected_pdf):
                file_size = os.path.getsize(expected_pdf) / (1024 * 1024)
                print(f" Already exists ({file_size:.1f} MB)")
                download_results[arxiv_id] = {
                    'success': True,
                    'pdf_path': expected_pdf,
                    'skipped': True
                }
                successful_downloads += 1
                if i % 20 == 0:
                    clear_output(wait=True)
                continue
            
            # Download the paper
            temp_config = {
                'download_from_arxiv': True,
                'arxiv_id': arxiv_id,
                'download_dir': download_dir,
                'existing_pdf': None
            }
            
            start_time = time.time()
            pdf_path, metadata = download_arxiv_paper_for_pipeline(temp_config)
            elapsed_time = time.time() - start_time
            
            if pdf_path:
                file_size = os.path.getsize(pdf_path) / (1024 * 1024)
                print(f" Downloaded ({file_size:.1f} MB, {elapsed_time:.1f}s)")
                download_results[arxiv_id] = {
                    'success': True,
                    'pdf_path': pdf_path,
                    'metadata': metadata,
                    'skipped': False
                }
                successful_downloads += 1
                
                # Small delay to be respectful to arXiv
                time.sleep(0.5)
                
            else:
                print(f" Failed ({elapsed_time:.1f}s)")
                download_results[arxiv_id] = {
                    'success': False,
                    'error': 'Download failed',
                    'skipped': False
                }

            if i % 5 == 0:  # Clear output every 5 downloads
                clear_output(wait=True)
                
        except Exception as e:
            print(f" Error: {str(e)[:50]}...")
            download_results[arxiv_id] = {
                'success': False,
                'error': str(e),
                'skipped': False
            }
    
    print(f"\n Download Summary:")
    print(f"  Total papers: {len(arxiv_ids)}")
    print(f"  Successfully downloaded: {successful_downloads}")
    print(f"  Failed: {len(arxiv_ids) - successful_downloads}")
    skipped = sum(1 for r in download_results.values() if r.get('skipped', False))
    print(f"  Already existed: {skipped}")
    print(f"  Newly downloaded: {successful_downloads - skipped}")
    
    return download_results

def process_paper_parallel(arxiv_id, pdf_info):
    """Process a single paper with OCR and analysis (for parallel execution)."""
    try:
        if not pdf_info['success']:
            return {
                'arxiv_id': arxiv_id,
                'success': False,
                'error': f"PDF not available: {pdf_info.get('error', 'Unknown')}"
            }
        
        # Run pipeline with existing PDF (skip download phase)
        result = run_complete_pipeline(
            arxiv_id=arxiv_id,
            pdf_path=pdf_info['pdf_path'],  # Use pre-downloaded PDF
            run_ocr=True,
            run_analysis=True,
            use_enhanced_ocr=False,
            use_annotations=False,
            use_robotics_prompts=USE_ROBOTICS_PROMPTS,  # Use the global setting
            force_reprocess=False,
            deepseek_runs=1,
            gemini_runs=0,
            temperature=0.1,
            verbose=False,  # Reduced verbosity for parallel processing
            max_chunk_size=16384,  # <<<-------  change this to adjust chunk size in parallel processing ---------
        )
        
        return {
            'arxiv_id': arxiv_id,
            'success': result['success'],
            'paper_id': result['paper_id'],
            'extractions': result['total_extractions'],
            'processing_time': result['processing_time'],
            'errors': len(result['errors']),
            'skipped_download': result['skipped_download'],
            'skipped_ocr': result['skipped_ocr'],
            'pdf_existed': pdf_info.get('skipped', False)
        }
        
    except Exception as e:
        return {
            'arxiv_id': arxiv_id,
            'success': False,
            'error': str(e)
        }


SETTING UP COMPLETE PIPELINE WITH PARALLEL PROCESSING...


In [10]:
def process_papers_sequentially_ocr_only(papers_dict, delay_seconds=2.0, verbose=True):
    """
    Process papers sequentially with OCR only to avoid rate limits.
    Use this after download_papers_sequentially() when parallel processing hits rate limits.
    
    Args:
        papers_dict: Dictionary from download_papers_sequentially() 
        delay_seconds: Delay between OCR requests to avoid rate limits
        verbose: Enable detailed logging
    
    Returns:
        Dictionary with processing results for each paper
    """
    if verbose:
        print(f"\n Sequential OCR Processing of {len(papers_dict)} papers")
        print(f"⏱️  Using {delay_seconds}s delay between requests to avoid rate limits")
        print("-" * 60)
    
    ocr_results = {}
    successful_ocr = 0
    
    for i, (arxiv_id, pdf_info) in enumerate(papers_dict.items(), 1):
        if verbose:
            print(f"\n Processing {i}/{len(papers_dict)}: {arxiv_id}")
        
        try:
            if not pdf_info['success']:
                ocr_results[arxiv_id] = {
                    'success': False,
                    'error': f"PDF not available: {pdf_info.get('error', 'Unknown')}",
                    'skipped_ocr': False,
                    'processing_time': 0
                }
                if verbose:
                    print(f"   Skipping - PDF not available")
                continue
            
            # Check if OCR already exists
            paper_identifier = arxiv_id
            ocr_output_dir = os.path.join("results", paper_identifier.replace('/', '_'))
            expected_md_name = f"{paper_identifier.replace('/', '_')}.md"
            expected_md_path = os.path.join(ocr_output_dir, expected_md_name)
            
            if os.path.exists(expected_md_path):
                file_size = os.path.getsize(expected_md_path)
                if file_size > 1000:  # At least 1KB
                    if verbose:
                        print(f"   OCR already exists ({file_size / 1024:.1f} KB) - skipping")
                      
                    ocr_results[arxiv_id] = {
                        'success': True,
                        'md_path': expected_md_path,
                        'skipped_ocr': True,
                        'processing_time': 0,
                        'existing_file_size': file_size
                    }
                    successful_ocr += 1
                    continue
                    
            
            # Run OCR processing
            if verbose:
                print(f"   Running OCR processing...")
            
            start_time = time.time()
            
            # Use the enhanced OCR processor
            md_path, json_path, annotations = process_pdf_with_enhanced_ocr(
                pdf_path=pdf_info['pdf_path'],
                output_dir=ocr_output_dir,
                api_key=ENHANCED_OCR_CONFIG['api_key'],
                api_endpoint=ENHANCED_OCR_CONFIG['api_endpoint'],
                max_pages=ENHANCED_OCR_CONFIG['max_annotation_pages'],
                include_images=ENHANCED_OCR_CONFIG['include_images'],
                use_annotations=ENHANCED_OCR_CONFIG['use_annotations'],
                verbose=False  # Reduce verbosity for sequential processing
            )
            
            processing_time = time.time() - start_time
            
            if md_path and json_path:
                # Check generated content
                with open(md_path, 'r', encoding='utf-8') as f:
                    content = f.read()
                
                ocr_results[arxiv_id] = {
                    'success': True,
                    'md_path': md_path,
                    'json_path': json_path,
                    'annotations': annotations,
                    'skipped_ocr': False,
                    'processing_time': processing_time,
                    'content_length': len(content),
                    'file_size': os.path.getsize(md_path)
                }
                successful_ocr += 1
                
                if verbose:
                    print(f"   OCR completed in {processing_time:.1f}s")
                    print(f"     Generated: {len(content):,} characters")
                    if annotations and ENHANCED_OCR_CONFIG['use_annotations']:
                        print(f"     Annotations: {annotations.get('total_annotations', 0)}")
            else:
                ocr_results[arxiv_id] = {
                    'success': False,
                    'error': 'OCR processing failed',
                    'skipped_ocr': False,
                    'processing_time': processing_time
                }
                if verbose:
                    print(f"   OCR failed after {processing_time:.1f}s")
            
            # Add delay to avoid rate limits (except for last item)
            if i < len(papers_dict) and delay_seconds > 0:
                if verbose:
                    print(f"  ⏱️  Waiting {delay_seconds}s before next request...")
                time.sleep(delay_seconds)
                
        except Exception as e:
            error_msg = f"Processing error: {str(e)}"
            ocr_results[arxiv_id] = {
                'success': False,
                'error': error_msg,
                'skipped_ocr': False,
                'processing_time': 0
            }
            if verbose:
                print(f"   Error: {str(e)[:100]}...")
    
    # Summary
    if verbose:
        print(f"\n Sequential OCR Summary:")
        print(f"  Total papers: {len(papers_dict)}")
        print(f"  Successfully processed: {successful_ocr}")
        print(f"  Failed: {len(papers_dict) - successful_ocr}")
        
        skipped_ocr = sum(1 for r in ocr_results.values() if r.get('skipped_ocr', False))
        newly_processed = successful_ocr - skipped_ocr
        
        print(f"  Already had OCR: {skipped_ocr}")
        print(f"  Newly processed: {newly_processed}")
        
        if newly_processed > 0:
            successful_new = [r for r in ocr_results.values() if r.get('success', False) and not r.get('skipped_ocr', False)]
            if successful_new:
                total_time = sum(r.get('processing_time', 0) for r in successful_new)
                avg_time = total_time / len(successful_new)
                print(f"  Average processing time: {avg_time:.1f}s per paper")
                print(f"  Total processing time: {total_time:.1f}s")
    
    return ocr_results

## LOAD THE PAPERS

In [11]:
# Main improved parallel processing
arxiv_ids_file = 'data/arxiv_ids/arxivIDs_SINGLE.txt'
arxiv_papers = import_arxiv_ids(arxiv_ids_file)
print(f"\n Starting improved parallel processing for {len(arxiv_papers)} arXiv papers...")

# Phase 1: Sequential Download
download_results = download_papers_sequentially(arxiv_papers)
# Filter successful downloads for processing
papers_to_process = {arxiv_id: info for arxiv_id, info in download_results.items() if info['success']}


ocr_results = process_papers_sequentially_ocr_only(
    papers_to_process, 
    delay_seconds=0.1,  # Adjust delay as needed
    verbose=True
)


 Imported 1 arXiv IDs from data/arxiv_ids/arxivIDs_SINGLE.txt
 Removed duplicates, 1 unique arXiv IDs available.

 Starting improved parallel processing for 1 arXiv papers...

 Phase 1: Sequential Download of 1 papers
------------------------------------------------------------


 Download Summary:
  Total papers: 1
  Successfully downloaded: 1
  Failed: 0
  Already existed: 1
  Newly downloaded: 0

 Sequential OCR Processing of 1 papers
⏱️  Using 0.1s delay between requests to avoid rate limits
------------------------------------------------------------

 Processing 1/1: 2412.19437
   OCR already exists (136.9 KB) - skipping

 Sequential OCR Summary:
  Total papers: 1
  Successfully processed: 1
  Failed: 0
  Already had OCR: 1
  Newly processed: 0


## RUN THE PIPELINE

In [12]:
if not papers_to_process:
    print("\n No papers available for processing. All downloads failed.")
else:
    print(f"\n Phase 2: Parallel Processing of {len(papers_to_process)} papers")
    print("-" * 60)
    
    # Phase 2: Parallel Processing (OCR + Analysis)
    parallel_results = []
    
    # Use fewer workers for processing to avoid overwhelming the system
    max_workers = min(16, len(papers_to_process))  # Reduced from 8 to 4
    print(f"Using {max_workers} parallel workers for processing...")
    
    #clear output every 5 papers to keep console clean
    clear_every = 5
    count = 0
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_arxiv = {
            executor.submit(process_paper_parallel, arxiv_id, pdf_info): arxiv_id 
            for arxiv_id, pdf_info in papers_to_process.items()
        }

        # clear output every 10 papers
        for future in future_to_arxiv:
            count += 1
            if count % clear_every == 0:
                clear_output(wait=True)
                print(f"Processing papers... {count}/{len(papers_to_process)} completed")
        
        for future in as_completed(future_to_arxiv):
            arxiv_id = future_to_arxiv[future]
            try:
                result = future.result()
                parallel_results.append(result)
                
                status = " SUCCESS" if result['success'] else " FAILED"
                skip_info = ""
                if result.get('pdf_existed'):
                    skip_info += " (PDF existed)"
                if result.get('skipped_ocr'):
                    skip_info += " (OCR skipped)"
                if result.get('skipped_download'):
                    skip_info += " (Download skipped)"
                
                extractions = result.get('extractions', 0)
                proc_time = result.get('processing_time', 0)
                print(f"{status}{skip_info} - {result['arxiv_id']} - {extractions} extractions in {proc_time:.1f}s")
                
            except Exception as e:
                print(f" PROCESSING FAILED for {arxiv_id} - {str(e)}")
                parallel_results.append({
                    'arxiv_id': arxiv_id,
                    'success': False,
                    'error': str(e)
                })

    # Final Summary
    print(f"\n Final Processing Summary:")
    print(f"="*60)
    print(f"Total papers requested: {len(arxiv_papers)}")
    print(f"Successfully downloaded: {len(papers_to_process)}")
    print(f"Successfully processed: {sum(1 for r in parallel_results if r.get('success', False))}")
    print(f"Failed processing: {sum(1 for r in parallel_results if not r.get('success', False))}")
    
    if parallel_results:
        successful_results = [r for r in parallel_results if r.get('success', False)]
        if successful_results:
            total_extractions = sum(r.get('extractions', 0) for r in successful_results)
            total_time = sum(r.get('processing_time', 0) for r in successful_results)
            avg_time = total_time / len(successful_results) if successful_results else 0
            
            print(f"\n Performance Metrics:")
            print(f"  Total extractions: {total_extractions}")
            print(f"  Total processing time: {total_time:.1f}s")
            print(f"  Average time per paper: {avg_time:.1f}s")
            

        else:
            print(f"\n No successful processing results found.")

print(f"\n Parallel processing completed!")



 Phase 2: Parallel Processing of 1 papers
------------------------------------------------------------
Using 1 parallel workers for processing...

=== Step 2: Token Analysis ===
 Document: results/2412.19437/2412.19437.md (136.85 KB)
Creating tokenizer for provider in tokenizer_factory.py: deepseek, model: None, max_tokens: 16384, overlap_tokens: 500, prompt_template_tokens: 1000, response_buffer_tokens: 1000, api_key: None
Using max_tokens: 16384 for provider: deepseek in tokenizer_factory.py
Initializing DeepSeekTokenizer for model 'deepseek-chat'
Max tokens set in initialization: 16384, Overlap tokens: 500
 Token Analysis:
  - Content length: 140,085 characters
  - Token count: 37,702 tokens
  - Tokenizer max: 16,384 tokens
  - Effective limit: 14,384 tokens
  - Chunking needed: Yes
  - Estimated chunks: 7
Creating ChunkedAnalyser with provider: deepseek, max_tokens: 16384, progressive: True
Initializing ChunkedAnalyser with provider: deepseek, max_tokens: 16384, overlap_tokens: 50

## ANALYSE THE PAPERS

In [ ]:
#function which counts the number of tokens using the tokenizer per paper in a given dictionary of papers coming from download_papers_sequentially()
def count_tokens_in_papers(papers_dict):
    print(papers_dict)
    tokenizer = get_tokenizer("deepseek", max_tokens=99999999999999)  # Use a very high max_tokens to avoid truncation
    print(" Tokenizer obtained successfully, counting tokens in papers...")
    if tokenizer:
        token_counts = {}
        for arxiv_id, info in papers_dict.items():
            print(f" Counting tokens for {arxiv_id}...")

            pdf_path = info.get('pdf_path')
            

            # Determine expected markdown file path
            paper_identifier = arxiv_id or Path(pdf_path).stem
           
            ocr_output_dir = os.path.join("results", paper_identifier.replace('/', '_'))
            
            # Check for existing markdown file
            expected_md_name = f"{paper_identifier.replace('/', '_')}.md"
            expected_md_path = os.path.join(ocr_output_dir, expected_md_name)
            
            if os.path.exists(expected_md_path):
                # Check file size to ensure it's not empty
                file_size = os.path.getsize(expected_md_path)
                if file_size > 1000:  # At least 1KB
                    existing_md_path = expected_md_path

                    print(f"  - Found existing markdown file: {expected_md_path}")
                    print(f"   Size: {file_size / 1024:.1f} KB")
                    print(f"   Counting tokens in markdown file...")
                    tmp_file = open(expected_md_path, 'r', encoding='utf-8')
                    prompt_length = tokenizer.count_tokens(tmp_file.read())
                    tmp_file.close()
                    print(f"  - Found {prompt_length} tokens in markdown file")
                    token_counts[arxiv_id] = prompt_length
                else:
                    print(f"  - Found existing markdown file but it's too small ({file_size} bytes), will reprocess")
                    existing_md_path = None
            else:
                print(f"  - No existing markdown file found at: {expected_md_path}")
           
                
        return token_counts

    else:
        print(" Failed to get tokenizer")
        return None
    
# count tokens from download_results
print("\n Counting tokens in downloaded papers...")
token_counts = count_tokens_in_papers(papers_to_process)


In [ ]:
if token_counts:
    for arxiv_id, count in token_counts.items():
        pass
       #print(f" {arxiv_id}: {count} tokens")
        
    #average tokens per paper
    avg_tokens = sum(token_counts.values()) / len(token_counts)
    print(f"Average tokens per paper: {avg_tokens:.2f}")
    #num of papers with less than 4k,8k,16k, 32k, 64k, 128k tokens
    num_less_than_4k = sum(1 for c in token_counts.values() if c < 4000)
    num_less_than_8k = sum(1 for c in token_counts.values() if c < 8000)
    num_less_than_16k = sum(1 for c in token_counts.values() if c < 16000)
    num_less_than_32k = sum(1 for c in token_counts.values() if c < 32000)
    num_less_than_64k = sum(1 for c in token_counts.values() if c < 64000)
    num_less_than_128k = sum(1 for c in token_counts.values() if c < 128000)
    print(f"Total number of papers: {len(token_counts)}")
    print(f"Number of papers with less than 4k tokens: {num_less_than_4k}")
    print(f"Number of papers with less than 8k tokens: {num_less_than_8k}")
    print(f"Number of papers with less than 16k tokens: {num_less_than_16k}")
    print(f"Number of papers with less than 32k tokens: {num_less_than_32k}")
    print(f"Number of papers with less than 64k tokens: {num_less_than_64k}")
    print(f"Number of papers with less than 128k tokens: {num_less_than_128k}")
    print(token_counts)

else:
    print(" Failed to count tokens.")

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

# Your example dictionary


# Step 1: Group lengths by 5k buckets
def group_by_5k(length):
    return (length // 5000) * 5000

grouped = [group_by_5k(length) for length in token_counts.values()]
bucket_counts = Counter(grouped)

# Step 2: Sort buckets for plotting
sorted_buckets = sorted(bucket_counts.items())

# Step 3: Unzip keys and counts
bucket_labels, counts = zip(*sorted_buckets)

# Step 4: Format labels for readability
labels = [f"{k}-{k+4999}" for k in bucket_labels]

# Step 5: Plot histogram
plt.figure(figsize=(10, 6))
plt.bar(labels, counts, width=0.8, align='center')
plt.xticks(rotation=45)
plt.xlabel("Token Length Range")
plt.ylabel("Count")
plt.title("Histogram of Token Lengths (Grouped by 5k)")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

token_lengths_values = list(token_counts.values())

# Plot histogram of log-transformed lengths
log_lengths = np.log10(token_lengths_values)

plt.hist(log_lengths, bins=20, edgecolor='black')
plt.xlabel("Log10(Token Length)")
plt.ylabel("Count")
plt.title("Log-Transformed Token Length Distribution")
plt.show()